# 2026 지옥캠프: 6공학관 vs IT관 분류 — 최종 제출

ImageNet 사전학습 EfficientNet-B0 4개를 결합하며, 전체 파라미터는 16,035,316개입니다.

- 데이터 분할 seed: 17 / 모델 seed: 17·18·19·20
- 입력: 192×192 / 검증·추론: 패딩 없는 중앙 크롭
- 최종 예측: 모델별 원본·좌우 반전 확률 평균 → 조건부 결합 → 온도 적용
- 제출 추론: 동봉된 `model.pt`와 `predict.py` 사용

## 대회 규칙 요약

| 항목 | 내용 |
|---|---|
| 과제 | 사진 → **6공학관(0) / IT관(1)** 이진 분류 |
| 참가 | 개인전 |
| 기간 | 2026-11-07(토) ~ 11-08(일) |
| 지표 | **Log Loss** (낮을수록 우수, 확률을 제출해야 함) |
| 모델 제약 | **총 파라미터 20M 이하** (앙상블·사전학습 백본 합산) |
| 런타임 | **Colab CPU 기준** 테스트 500장 10분 이내 · 실행 중 다운로드 금지 (사전학습 가중치는 최종 파일에 포함) |
| 금지 | 외부 이미지 사용, 테스트 이미지를 학습에 사용(pseudo-labeling 포함) |
| 제출 | 예측 CSV **10분에 1회 · 하루 최대 50회** (공개 리더보드) + 마감 시 최종 파일 1개 (형식 자유) |
| 순위 | 테스트 500장 중 Public 250장 = 공개 리더보드, **Private 250장 = 최종 순위** (어느 장이 Private 인지는 비공개) |


## 실행 준비

### 제출 가중치로 추론

`model.py`, `predict.py`, `model.pt`를 같은 폴더에 두고 실행합니다. 추론 중 가중치를 다운로드하지 않습니다.

```shell
python predict.py --input <테스트 이미지 폴더> --output pred.csv
```

추론 의존성은 `torch`, `torchvision`, `Pillow`입니다.

### 학습 재현

별도 작업 폴더에 `baseline.ipynb`, `train/6e`, `train/it`, `test`, `sample_submission.csv`를 준비한 뒤 전체 실행합니다. 학습에는 `numpy`, `scikit-learn`, `matplotlib`도 필요합니다.

학습 시 ImageNet 초기 가중치를 다운로드할 수 있습니다. 새 체크포인트·CSV·패키지는 `RESULT_DIR` 아래에 저장합니다. 노트북에서 생성하는 `model.py`와 `predict.py`는 작업 폴더에 저장합니다.

재학습 결과는 하드웨어·라이브러리에 따라 달라질 수 있습니다. 제출용 CSV의 완전 일치는 동봉된 가중치로 별도 확인합니다. 마지막 자가 검증 셀은 운영진 원본과 같이 노트북과 패키지 추론 확률을 허용 오차 내에서 비교합니다.


## 모듈 불러오기 및 시드 고정

Python·NumPy·PyTorch 시드를 고정합니다. 각 모델은 생성 전과 학습 직전에 시드를 다시 고정합니다.


In [ ]:
import csv
import importlib
import os
import shutil
import subprocess
import sys
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 17
BASE_DIR = Path.cwd().resolve()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(SEED)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
	print(torch.cuda.get_device_name())


## 하이퍼 파라미터

In [ ]:
params = {
	'epoch': 50,
	'early_stopping': True,
	'min_epochs': 5,
	'patience': 5,
	'batch_size': 32,
	'learning_rates': (0.001, 0.0003, 0.0001),
	'valid_ratio': 0.15,
	'num_show_img': 5,
}


In [ ]:
class_names = {0: '6e', 1: 'it'}
RUN_SEEDS = tuple(range(SEED, SEED + 4))
AUGMENTATION_BY_SEED = dict(zip(RUN_SEEDS, ('square', 'rectangle', 'square_erasing', 'square_perspective')))
RESULT_DIR = BASE_DIR / '실험결과' / '011_192크롭_패딩제거' / f'seed_{SEED}'
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
EPOCH_DIR = RESULT_DIR / 'epochs'
assert not ((RESULT_DIR / 'runs.csv').exists() or (RESULT_DIR / 'model.pt').exists() or any(CHECKPOINT_DIR.glob('*.pt'))), f'기존 학습 결과가 있습니다: {RESULT_DIR}'
for directory in (CHECKPOINT_DIR, EPOCH_DIR):
	directory.mkdir(parents=True, exist_ok=True)
print('결과 위치:', RESULT_DIR)


## 모델 정의 (`model.py` 생성)

아래 셀은 모델 구조, 크롭 전처리와 앙상블 결합 함수를 `model.py`로 저장합니다. 셀을 수정하면 다시 실행해야 파일에 반영됩니다.


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as F

IMG_SIZE = 192
MIN_CROP_SIZE = 96
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PERSPECTIVE_FILL = tuple(round(value * 255) for value in NORM_MEAN)
TEMPERATURE_RULES = ("T0.3", "T0.5")


class RandomPhotoCrop:
	def __init__(self, square):
		self.square = square

	def __call__(self, image):
		width, height = image.size
		assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작은 원본입니다: {image.size}"
		crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		if self.square:
			crop_height = crop_width
		else:
			crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
			while (crop_width / crop_height < 0.75) or (crop_width / crop_height > 4 / 3):
				crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
				crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		top = int(torch.randint(height - crop_height + 1, ()).item())
		left = int(torch.randint(width - crop_width + 1, ()).item())
		return F.crop(image, top, left, crop_height, crop_width)


def center_crop(image):
	width, height = image.size
	assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작아 패딩 없이 크롭할 수 없습니다: {image.size}"
	return F.center_crop(image, [IMG_SIZE, IMG_SIZE])


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(center_crop),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs, temperature_rule="T0.3"):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 4)
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	middle_mean = member_probs.sort(dim=1).values[:, 1:3].mean(dim=1, keepdim=True)
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), middle_mean)
	if temperature_rule == "T0.3":
		temperatures = torch.where(unanimous, 0.3, 0.7)
	elif temperature_rule == "T0.5":
		temperatures = 0.5
	else:
		raise ValueError(f"알 수 없는 temperature 규칙: {temperature_rule}")
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(pretrained=False):
	weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
	net = models.efficientnet_b0(weights=weights)
	net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	return net


class EfficientNetEnsemble(nn.Module):
	def __init__(self, state_dicts):
		super().__init__()
		assert len(state_dicts) == 4, "앙상블 가중치 4개가 필요합니다."
		self.models = nn.ModuleList()
		for state_dict in state_dicts:
			model = build_model()
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs):
		return torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)

	def forward(self, inputs, temperature_rule="T0.3"):
		return combine_probabilities(self.member_probabilities(inputs), temperature_rule)


In [ ]:
import model as model_module
importlib.invalidate_caches()
importlib.reload(model_module)
assert Path(model_module.__file__).resolve() == BASE_DIR / 'model.py'
model = model_module.build_model()
print('EfficientNet-B0 / input:', model_module.IMG_SIZE)


### 파라미터 수 확인

단일 모델과 네 모델의 합계를 출력합니다. 실제 앙상블을 구성한 뒤 20,000,000개 제한을 검사합니다.


In [ ]:
PARAM_LIMIT = 20_000_000
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f'1모델 {n_params:,} / 4모델 {4 * n_params:,} / 제한 {PARAM_LIMIT:,}')
del model


## 데이터 불러오기 및 증강

학습 이미지는 각 변 96~192픽셀을 크롭한 뒤 192×192로 확대합니다. 모델마다 정사각형·직사각형·지우기·원근 변환 증강을 사용합니다.

검증·추론은 192×192 중앙 크롭이며 크기 맞춤 패딩을 사용하지 않습니다. 원근 변환의 모서리 채움은 유지합니다. 모든 입력은 ImageNet 평균과 표준편차로 정규화합니다.


In [ ]:
data_path = BASE_DIR / 'train'
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD
AUGMENTATION_SETTINGS = {
	'square': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.0},
	'rectangle': {'square': False, 'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.3, 'erase_p': 0.0, 'perspective_p': 0.0},
	'square_erasing': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.1, 'perspective_p': 0.0},
	'square_perspective': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.3},
}
train_transforms = {}
for name, settings in AUGMENTATION_SETTINGS.items():
	operations = [
		model_module.RandomPhotoCrop(square=settings['square']),
		transforms.RandomHorizontalFlip(p=0.5),
		transforms.ColorJitter(brightness=settings['brightness'], contrast=settings['contrast'], saturation=settings['saturation']),
	]
	if settings['perspective_p'] > 0:
		operations.append(transforms.RandomPerspective(distortion_scale=0.15, p=settings['perspective_p'], fill=model_module.PERSPECTIVE_FILL))
	operations.extend([
		transforms.Resize((IMG_SIZE, IMG_SIZE)),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])
	if settings['erase_p'] > 0:
		operations.append(transforms.RandomErasing(p=settings['erase_p'], scale=(0.02, 0.1), value=0))
	train_transforms[name] = transforms.Compose(operations)
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transforms['square'])
full_valid = datasets.ImageFolder(data_path, valid_transform)
assert full_train.class_to_idx == {'6e': 0, 'it': 1}
print('클래스:', full_train.class_to_idx, '/ 전체:', len(full_train), '/ 입력:', IMG_SIZE)


### 학습·검증 분할

클래스 비율을 유지해 학습 85%, 검증 15%로 분할합니다. 검증 이미지는 학습에 사용하지 않으며, 분할 목록은 `validation_split.csv`로 저장합니다.


In [ ]:
labels = [label for _, label in full_train.samples]
train_idx, valid_idx = train_test_split(list(range(len(full_train))), test_size=params['valid_ratio'], stratify=labels, random_state=SEED)
data_sets = {'train': Subset(full_train, train_idx), 'valid': Subset(full_valid, valid_idx)}
with (RESULT_DIR / 'validation_split.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'label'))
	writer.writerows((Path(full_valid.samples[index][0]).stem, labels[index]) for index in valid_idx)
print(f'train: {len(train_idx)} / valid: {len(valid_idx)} / split seed: {SEED}')


In [ ]:
dataloaders = {
	'train': DataLoader(data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED)),
	'valid': DataLoader(data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
}
print('배치 수:', len(dataloaders['train']), len(dataloaders['valid']))


### 데이터 미리보기

미리보기 이후 모델 생성 전과 학습 직전에 시드를 다시 고정하므로, 미리보기가 학습 난수 흐름에 영향을 주지 않습니다.


In [ ]:
def img_show(tensor, title=None):
	arr = tensor.numpy().transpose((1, 2, 0))
	arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
	arr = np.clip(arr, 0, 1)
	plt.figure(figsize=(12, 4))
	plt.imshow(arr)
	plt.axis('off')
	if title is not None:
		plt.title(title)
	plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])


## 학습

네 모델을 각각 학습하며, `BCEWithLogitsLoss`로 검증 손실을 계산합니다. 검증 손실이 가장 낮은 가중치를 모델별 체크포인트에 저장하고, 손실이 같으면 정확도가 높은 가중치를 선택합니다.

학습률은 0.001 → 0.0003 → 0.0001 순서입니다. 각 단계에서 최소 5 epoch를 진행하며, 이후 검증 손실이 5회 연속 개선되지 않으면 다음 학습률로 전환합니다. 마지막 단계에서는 조기 종료하며 전체 최대 50 epoch입니다.

이 검증 손실은 개별 모델의 값이며, 최종 앙상블에 TTA와 온도를 적용한 Log Loss 및 리더보드 점수와 구분합니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
	"""optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
	training = optimizer is not None
	model.train(training)

	total_loss, total_correct, total_n = 0.0, 0, 0
	with torch.set_grad_enabled(training):
		for inputs, targets in loader:
			inputs = inputs.to(device)
			targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

			logits = model(inputs)
			loss = criterion(logits, targets)
			assert bool(torch.isfinite(loss)), "학습/검증 손실이 비유한 값입니다."

			if training:
				optimizer.zero_grad()
				loss.backward()
				optimizer.step()

			probs = torch.sigmoid(logits)
			preds = (probs >= 0.5).float()
			total_loss += loss.item() * inputs.size(0)
			total_correct += (preds == targets).sum().item()
			total_n += inputs.size(0)

	return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates, pt_file_name):
	assert (min_epochs > 0) and (patience > 0) and (len(learning_rates) > 0) and all(rate > 0 for rate in learning_rates), '학습 단계 설정은 양수여야 합니다.'
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}', flush=True)

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
results = []
histories = {}

for seed in RUN_SEEDS:
	augmentation = AUGMENTATION_BY_SEED[seed]
	full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
	run_data_sets = {'train': Subset(full_run_train, train_idx), 'valid': data_sets['valid']}
	print(f'\n학습 시작: seed {seed} / {augmentation}', flush=True)
	seed_everything(seed)
	model = model_module.build_model(pretrained=True).to(device)
	seed_everything(seed)
	dataloaders = {
		'train': DataLoader(run_data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed)),
		'valid': DataLoader(run_data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
	}
	criterion = nn.BCEWithLogitsLoss()
	optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rates'][0])
	pt_file_name = CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt'
	started = time.perf_counter()
	with (EPOCH_DIR / f'efficientnet_b0_seed{seed}.csv').open('w', newline='', encoding='utf-8-sig') as epoch_csv:
		history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params['epoch'], epoch_csv=epoch_csv, early_stopping=params['early_stopping'], min_epochs=params['min_epochs'], patience=params['patience'], learning_rates=params['learning_rates'], pt_file_name=pt_file_name)
	elapsed = time.perf_counter() - started
	histories[seed] = (history, best_epoch, best_loss)
	results.append({
		'model': 'efficientnet_b0', 'seed': seed, 'augmentation': augmentation,
		'image_size': IMG_SIZE, 'selection_metric': 'valid_loss', 'learning_rates': str(params['learning_rates']),
		'max_epochs': params['epoch'], 'epochs_run': len(history['train_loss']),
		'early_stopping': params['early_stopping'], 'min_epochs': params['min_epochs'], 'patience': params['patience'],
		'best_epoch': best_epoch, 'valid_loss': best_loss, 'valid_acc': history['valid_acc'][best_epoch - 1],
		'seconds': elapsed, 'n_params': n_params, 'checkpoint': str(pt_file_name),
	})
	with (RESULT_DIR / 'runs.csv').open('w', newline='', encoding='utf-8-sig') as output:
		writer = csv.DictWriter(output, fieldnames=list(results[0]))
		writer.writeheader()
		writer.writerows(results)
	del model, optimizer, dataloaders
	if device.type == 'cuda':
		torch.cuda.empty_cache()


### 학습 곡선

모델별 학습·검증 손실과 정확도, 선택된 epoch를 표시합니다.


In [ ]:
for seed in RUN_SEEDS:
	history, best_epoch, best_loss = histories[seed]
	epochs = range(1, len(history['train_loss']) + 1)
	fig, axes = plt.subplots(1, 2, figsize=(12, 4))
	for axis, metric in zip(axes, ('loss', 'acc')):
		axis.plot(epochs, history[f'train_{metric}'], label='train')
		axis.plot(epochs, history[f'valid_{metric}'], label='valid')
		axis.set_xlabel('epoch')
		axis.set_ylabel(metric)
		axis.legend()
		axis.grid(alpha=0.3)
	fig.suptitle(f'seed {seed} / selected epoch {best_epoch} / loss {best_loss:.6f}')
	fig.tight_layout()
	plt.show()
	plt.close(fig)


## 테스트 추론 및 `submission.csv` 생성

모델별 최적 체크포인트를 하나의 `model.pt`에 저장한 뒤 CPU에서 추론합니다.

1. 각 모델에서 원본·좌우 반전 이미지의 sigmoid 확률을 평균합니다.
2. 네 모델의 분류가 모두 일치하면 전체 확률 평균에 T=0.3을 적용합니다.
3. 불일치하면 정렬한 가운데 두 확률의 평균에 T=0.7을 적용합니다.

최종 확률은 IT관(label 1)일 확률이며, 파일명 순서와 소수점 6자리 형식으로 `RESULT_DIR/submission.csv`에 저장합니다.


In [ ]:
class TestImageDataset(Dataset):
	def __init__(self, files, transform):
		self.files = files
		self.transform = transform

	def __len__(self):
		return len(self.files)

	def __getitem__(self, index):
		path = self.files[index]
		with Image.open(path) as image:
			return self.transform(image.convert('RGB')), path.stem


test_files = sorted(path for path in (BASE_DIR / 'test').iterdir() if path.is_file() and (path.suffix.lower() in ('.jpg', '.jpeg', '.png')))
with (BASE_DIR / 'sample_submission.csv').open(newline='', encoding='utf-8-sig') as source:
	sample_rows = list(csv.DictReader(source))
expected_ids = {row['id'] for row in sample_rows}
assert (len(test_files) == 500) and (len(expected_ids) == 500) and (len({path.stem for path in test_files}) == 500)
assert {path.stem for path in test_files} == expected_ids
test_loader = DataLoader(TestImageDataset(test_files, valid_transform), batch_size=params['batch_size'], shuffle=False, num_workers=0)


In [ ]:
checkpoint = {
	'split_seed': SEED, 'image_size': IMG_SIZE, 'preprocessing': 'center_crop_192_no_padding',
	'seeds': list(RUN_SEEDS), 'augmentations': [AUGMENTATION_BY_SEED[seed] for seed in RUN_SEEDS],
	'state_dicts': [torch.load(CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt', map_location='cpu', weights_only=True) for seed in RUN_SEEDS],
}
torch.save(checkpoint, RESULT_DIR / 'model.pt')
ensemble = model_module.EfficientNetEnsemble(checkpoint['state_dicts']).eval()
ensemble_params = sum(parameter.numel() for parameter in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT
print('모델 저장:', RESULT_DIR / 'model.pt')

rows = []
with torch.inference_mode():
	for inputs, image_ids in test_loader:
		members = ensemble.member_probabilities(inputs)
		flipped_members = ensemble.member_probabilities(inputs.flip(dims=[3]))
		probs = model_module.combine_probabilities((members + flipped_members) / 2, 'T0.3').squeeze(1)
		assert bool(torch.isfinite(probs).all())
		rows.extend((image_id, f'{prob.item():.6f}') for image_id, prob in zip(image_ids, probs))
assert (len(rows) == 500) and ({row[0] for row in rows} == expected_ids)
output_path = RESULT_DIR / 'submission.csv'
with output_path.open('w', newline='', encoding='utf-8') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'prob'))
	writer.writerows(rows)
print('예측 저장:', output_path)


## 최종 파일 생성 (예시: `predict.py` 패키지)

마감 후 운영진이 **모든 참가자의 최종 파일**을 받아 **Colab CPU** 환경에서 직접 실행합니다.

1. 공개 리더보드(Public) 점수가 재현되는지 확인
2. 런타임(테스트 500장 10분 이내)과 파라미터 수(20M 이하) 확인
3. 통과한 참가자만 Private 250장 점수로 최종 순위 산정

**제출 형식은 자유**입니다. 운영진이 실행할 수 있고 가중치가 포함되어 있으면 됩니다. 아래는 이 노트북이 만드는 예시 형식이며, 이렇게 내면 다음 명령 한 줄로 검증됩니다.

```shell
python predict.py --input <이미지 폴더> --output pred.csv
```

패키지 구성:
```
📦submission_package.zip
 ├─📜model.py     # 모델 구조 정의 (build_model 함수 필수)
 ├─📜predict.py   # 추론 진입점 (--input/--output 인터페이스 유지)
 └─📜model.pt     # 학습된 가중치 (사전학습 가중치 사용 시에도 여기에 포함)
```

### 이 버전의 패키지 구성
새 압축 파일은 생성하지 않습니다. RESULT_DIR/package/에 model.py, predict.py, model.pt, baseline.ipynb를 복사합니다.
기본 predict.py 명령은 추가 옵션 없이 T0.3 TTA를 사용합니다. 추론에서는 pretrained=False로 모델 구조를 만들고 동봉된 전체 가중치를 불러오므로 다운로드하지 않습니다.
학습 노트북은 학습 재현용이며 평가 시에는 predict.py만 실행하면 됩니다.


In [ ]:
%%writefile predict.py
import argparse
import csv
from pathlib import Path

import torch
from PIL import Image

from model import IMG_SIZE, TEMPERATURE_RULES, EfficientNetEnsemble, build_valid_transform, combine_probabilities

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
	files = sorted(path for path in Path(input_dir).iterdir() if path.is_file() and (path.suffix.lower() in (".jpg", ".jpeg", ".png")))
	assert files, f"이미지가 없습니다: {input_dir}"
	assert len({path.stem for path in files}) == len(files), "이미지 ID가 중복됩니다."
	return files


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--temperature-rule", choices=TEMPERATURE_RULES, default="T0.3")
	parser.add_argument("--tta", action="store_true", default=True, help="모델별 원본·좌우 반전 확률 평균 후 결합")
	args = parser.parse_args()
	weights_path = Path(args.weights)
	if not weights_path.is_absolute():
		weights_path = Path(__file__).resolve().parent / weights_path
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE
	assert checkpoint["preprocessing"] == "center_crop_192_no_padding"
	model = EfficientNetEnsemble(checkpoint["state_dicts"]).eval()
	n_params = sum(parameter.numel() for parameter in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	transform = build_valid_transform()
	files = load_images(args.input)
	rows = []
	with torch.inference_mode():
		for position in range(0, len(files), 32):
			batch_files = files[position:position + 32]
			images = []
			for path in batch_files:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images)
			member_probs = model.member_probabilities(inputs)
			if args.tta:
				member_probs = (member_probs + model.member_probabilities(inputs.flip(dims=[3]))) / 2
			probs = combine_probabilities(member_probs, args.temperature_rule).squeeze(1)
			assert bool(torch.isfinite(probs).all())
			for path, prob in zip(batch_files, probs):
				rows.append((path.stem, f"{prob.item():.6f}"))
	with Path(args.output).open("w", newline="", encoding="utf-8") as output:
		writer = csv.writer(output)
		writer.writerow(("id", "prob"))
		writer.writerows(rows)
	print(f"{args.output} 저장 완료 / {len(files)}장 / 파라미터 {n_params:,}개 / TTA={args.tta}")


if __name__ == "__main__":
	main()


In [ ]:
package_dir = RESULT_DIR / 'package'
assert not package_dir.exists(), f'기존 패키지를 덮어쓰지 않습니다: {package_dir}'
package_dir.mkdir()
for source, destination in (
	(BASE_DIR / 'model.py', 'model.py'),
	(BASE_DIR / 'predict.py', 'predict.py'),
	(RESULT_DIR / 'model.pt', 'model.pt'),
	(BASE_DIR / 'baseline.ipynb', 'baseline.ipynb'),
):
	shutil.copy2(source, package_dir / destination)
package_bytes = sum(path.stat().st_size for path in package_dir.iterdir() if path.is_file())
assert package_bytes <= 200_000_000, f'패키지 크기 초과: {package_bytes:,}'
print('패키지 폴더:', package_dir)
print(f'전체 크기: {package_bytes:,} bytes / 압축 생성 없음')


### 패키지 자가 검증 (제출 전 필수!)

운영진 검증과 같은 방식으로, 패키지 폴더 안에서 `predict.py`를 CPU로 실행해 봅니다. 여기서 에러가 나면 검증도 실패합니다. Colab 런타임을 CPU로 두고 실행하면 10분 기준도 함께 확인할 수 있습니다.


In [ ]:
result = subprocess.run(
	[sys.executable, '-B', 'predict.py', '--input', str(BASE_DIR / 'test'), '--output', str(RESULT_DIR / 'package_check.csv')],
	cwd=package_dir, capture_output=True, text=True, encoding='utf-8',
	env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'predict.py 실행 실패! 패키지를 점검하세요.'

# 노트북 추론 결과와 패키지 추론 결과가 (허용 오차 내에서) 일치하는지 확인
# GPU로 학습·추론한 경우 CPU 추론과 부동소수점 미세 차이가 나는 것은 정상입니다
def read_probs(path):
	with open(path) as f:
		next(f)  # header
		return {line.split(',')[0]: float(line.split(',')[1]) for line in f if line.strip()}

nb_probs = read_probs(RESULT_DIR / 'submission.csv')
pkg_probs = read_probs(RESULT_DIR / 'package_check.csv')
assert nb_probs.keys() == pkg_probs.keys(), '파일 목록이 다릅니다!'
max_diff = max(abs(nb_probs[k] - pkg_probs[k]) for k in nb_probs)
print(f'노트북 vs 패키지 확률 최대 차이: {max_diff:.2e}')
assert max_diff < 1e-3, '차이가 너무 큽니다 — 전처리나 가중치가 어긋났는지 확인하세요.'
print('패키지 자가 검증 통과 ✓')


## 제출 전 체크리스트

- [ ] `submission.csv`가 `id,prob` 형식이고 테스트 500장 전부를 포함하는가?
- [ ] 파라미터 카운터 셀에서 **20M 이하**를 확인했는가? (앙상블·사전학습 백본 합산)
- [ ] 패키지 자가 검증 셀이 에러 없이 통과했는가? (Colab CPU 에서 500장 10분 이내)
- [ ] `predict.py`가 네트워크 없이 동작하는가? (다운로드 코드 금지, 가중치는 패키지에 동봉)
- [ ] 테스트 이미지를 학습에 사용하지 않았는가? (pseudo-labeling 포함 금지)
- [ ] 외부 이미지를 학습에 사용하지 않았는가?
